In [3]:
"""
Created on Mon Mar 10 2025
@author: Max Van Migem
"""
import numpy as np
import os
import time
import sys
import pickle
from psychopy import parallel, visual, gui, data, event, core, monitors
from psychopy.visual import ShapeStim
from psychopy import logging
from math import fabs
rng = np.random.default_rng(seed=None)
linejitter_arr =(np.ones((30,30,2))- rng.random((30,30,2))*2)/80
# file_name = os.getcwd() + '/stim_jitter.npy'
# np.save(file_name,linejitter_arr)

ModuleNotFoundError: No module named 'psychopy'

In [ ]:
def generateStimCoordinates(gridcol, gridrow, jitter):
    """
    Generates coordinates used to draw the stimulus lines
    """
    # Calculate spacing of grid
    x_spacing = 1.0 / (gridcol)
    y_spacing = 1.0 / (gridrow)

    # Create an array to store coordinates: every point (x,y) of the grid for every quadrant (4) 
    coord_array = np.empty(shape = (gridcol,gridrow,2,4),dtype= 'object')
    quadrant_set = [[-1,1],[1,1],[1,-1],[-1,-1]]

    # Generate grid coord per quadrant
    for i,quad in enumerate(quadrant_set):
        # Per row
        for row in range(gridrow):
            # Per column
            for col in range (gridcol):

                grid_x = col * x_spacing  # grid points
                grid_y = row * y_spacing  

                grid_x = grid_x * quad[0]   # quadrant position
                grid_y = grid_y * quad[1]   # quadrant position

                coord_array[col,row,0,i] = grid_x  # Store them in big ass array
                coord_array[col,row,1,i] = grid_y

    # flip the matrices
    coord_array[:,:,0,0] = np.flip(coord_array[:,:,0,0], axis=0) 
    coord_array[:,:,1,0] = np.flip(coord_array[:,:,1,0], axis=0) 

    coord_array[:,:,0,2] = np.flip(coord_array[:,:,0,2], axis=1) 
    coord_array[:,:,1,2] = np.flip(coord_array[:,:,1,2], axis=1) 

    coord_array[:,:,0,3] = np.flip(coord_array[:,:,0,3], axis=(0,1)) 
    coord_array[:,:,1,3] = np.flip(coord_array[:,:,1,3], axis=(0,1)) 

    #  Add jitter
    for i in range(4):
        coord_array[:,:,:,i] =  coord_array[:,:,:,i] + jitter[:gridcol,:gridrow,:]
  
    return coord_array

In [43]:
import numpy as np
def generatePredictionList(tr_block, n_odd, odd, mode = 'rotation'):
    """ 
    Generate pseudo randomized properties of trial direction or stimulus angle on a block level
    Parameters:
        tr_block (int):number of trials per block
        n_odd (int): number of odd per block
        odd (int): indicate which direction or angle is odd 0 clockwise/left and 1 anticlockwise/right
        mode (str): indentifies whether it is trial level ('rotation) or stimulus ('angle') 
    """
    # Check if there are correct amount of odds
    if n_odd > tr_block/2:
        raise ValueError("Invalid input parameters")
    # Multiple the trial and odd var by 4 because there are 4 stimuli per trial
    if mode == 'angle': 
        n_odd = n_odd*4
        tr_block = tr_block*4

    prediction_arr = np.zeros(tr_block , dtype=int)
    possible_positions = np.arange(tr_block)
    # Randomly select positions for 1's, ensuring no two are consecutive
    selected_positions = []
    while len(selected_positions) < n_odd:
        pos = np.random.choice(possible_positions)
        selected_positions.append(pos)
        # Remove the selected position and its adjacent positions to prevent consecutive 1's
        possible_positions = possible_positions[(possible_positions < pos - 1) | (possible_positions > pos + 1)]
    prediction_arr[selected_positions] = 1
    
    if not odd:
        prediction_arr = prediction_arr ^ 1
        
    return prediction_arr

In [9]:
def participantCounterBalance(participant_number,conds = 4):
    """ 
    Assign particpant to conditions such as key-mappings and odd vs regular direction/angles
    Parameters:
        participant_number (int):identifier that serves as unique key for assignment
        conds (int): number of conditions
    Returns:
        counter_code (ndarray): array with length conds with a code for every participant 
    """
    counter_code = np.empty(conds, dtype=int)
    for i in range(conds):
        counter_code[i] = (participant_number // (i+1)) % 2

    return counter_code

In [50]:
arr_o = generatePredictionList(30,n_odd=7,odd = 0)

In [ ]:
for i in range(20):
    print(participantCounterBalance(i))

In [61]:
# Participant specific values
rotodd_map = ['clockwise','anticlockwise']
angodd_map = ['left','right']
key_map = [['k','d'],['d','k']]

subject_code = participantCounterBalance(2)  
rotation_odd = rotodd_map[subject_code[0]] # index 0 idicate which direction is odd
angle_odd = angodd_map[subject_code[1]] # index 1 indicates which angle is odd
rotation_keys = key_map[subject_code[2]] # index 2 indicates which keys are paired to the directions
angle_keys = key_map[subject_code[3]] # index 3 indicates which keys are paired to the angles

key_mappings = {rotodd_map[0]:rotation_keys[0],rotodd_map[1]:rotation_keys[1],
                angodd_map[0]:angle_keys[0],angodd_map[1]:angle_keys[1]}

In [59]:
key_mappings['left']

'd'